In [ ]:
import os
import sys
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import lightning as L
from torchvision.models import resnet50, ResNet50_Weights
import torchmetrics
from lightning.pytorch.loggers import WandbLogger
import wandb
from dotenv import load_dotenv
import torch.nn.functional as F
from tqdm.notebook import tqdm
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from torch.utils.data import DataLoader
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'

for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from classification_pipeline import (
    load_or_download_classification_dataset,
    ClassificationDownloadConfig,
    build_multiclass_classification_records_from_masks,
    split_grouped_records,
    ToothCropDataset,
    build_classification_resize_pipeline
)

In [ ]:
class ToothClassificationDataModule(L.LightningDataModule):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage: str | None = None):
        _, coco_data, image_dirs = load_or_download_classification_dataset(
            ClassificationDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download
        )
        
        all_records, self.label_map = build_multiclass_classification_records_from_masks(
            coco_data, image_dirs, crop_margin=0.08
        )

        caries_idx = self.label_map.get('Caries', 44)

        for rec in all_records:
            if rec['label'] == caries_idx:
                rec['label'] = 1
            else:
                rec['label'] = 0
                
        self.label_map = {'No_Caries': 0, 'Caries': 1}
        
        train_rec, val_rec, test_rec = split_grouped_records(
            all_records, train_size=0.7, val_size=0.15, test_size=0.15
        )

        all_labels = [rec['label'] for rec in all_records]
        num_classes = len(self.label_map)
        total_samples = len(all_labels)

        weights = []
        for i in range(num_classes):
            count = all_labels.count(i)

            weight = total_samples / (num_classes * count) if count > 0 else 0.0
            weights.append(weight)

        self.class_weights = torch.tensor(weights, dtype=torch.float)

        print(f"Number of classes: {num_classes}")
        print(f'Class weights: {self.class_weights}')

        self.train_ds = train_rec
        self.val_ds = val_rec
        self.test_ds = test_rec
        
        aug_pipeline = build_classification_image_pipeline()
        resize_pipeline = build_classification_resize_pipeline(self.cfg.image_size)
        
        self.train_ds = ToothCropDataset(
            self.train_ds,
            image_size=self.cfg.image_size,
            image_transform=aug_pipeline,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        self.val_ds = ToothCropDataset(
            self.val_ds,
            image_size=self.cfg.image_size,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        self.test_ds = ToothCropDataset(
            self.test_ds,
            image_size=self.cfg.image_size,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        
        print(f'Train samples: {len(self.train_ds)}\nVal samples: {len(self.val_ds)}\nTest samples: {len(self.test_ds)}')
    
    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            **self._loader_kwargs()
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs()
        )
        
    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs()
        )

In [6]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class LitToothClassifier(L.LightningModule):
    def __init__(self, cfg, class_weights=None):
        super().__init__()
        self.cfg = cfg

        self.model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, 2)

        self.accuracy = torchmetrics.Accuracy(task="binary")
        self.f1 = torchmetrics.F1Score(task="binary")

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        self.log("train/loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        preds = torch.argmax(logits, dim=1)
        self.accuracy(preds, y)
        self.f1(preds, y)

        self.log("val/loss", loss, prog_bar=True, on_epoch=True)
        self.log("val/acc", self.accuracy, prog_bar=True, on_epoch=True)
        self.log("val/f1", self.f1, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        
        loss = F.cross_entropy(logits, y)
        
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        
        self.accuracy(preds, y)
        self.f1(preds, y)
        
        self.log('test/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test/acc", self.accuracy, on_step=False, on_epoch=True)
        self.log("test/f1", self.f1, on_step=False, on_epoch=True)
        
        if not hasattr(self, 'test_outputs'):
            self.test_outputs = []
            
        self.test_outputs.append({
            'probs': probs.cpu().numpy(), 
            'labels': y.cpu().numpy(), 
            'preds': preds.cpu().numpy()
        })
        return loss

    def on_test_epoch_end(self):
        if not hasattr(self, 'test_outputs') or len(self.test_outputs) == 0:
            return

        all_probs = np.concatenate([x['probs'] for x in self.test_outputs], axis=0)
        all_labels = np.concatenate([x['labels'] for x in self.test_outputs], axis=0)
        all_preds = np.concatenate([x['preds'] for x in self.test_outputs], axis=0)
        
        class_names = ['No_Caries', 'Caries']
        n_classes = 2
        colors = ['green', 'red']
        
        y_bin = np.zeros((len(all_labels), n_classes))
        y_bin[np.arange(len(all_labels)), all_labels] = 1
        
        metrics_to_log = {}
        
        fig_curves, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
        
        for i, color in zip(range(n_classes), colors):
            fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
            roc_auc = auc(fpr, tpr)
            ax1.plot(fpr, tpr, color=color, lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
            metrics_to_log[f'test/auc_roc_{class_names[i].lower()}'] = roc_auc
            
            precision, recall, _ = precision_recall_curve(y_bin[:, i], all_probs[:, i])
            pr_auc = average_precision_score(y_bin[:, i], all_probs[:, i])
            ax2.plot(recall, precision, color=color, lw=2, label=f'{class_names[i]} (AUC = {pr_auc:.2f})')
            metrics_to_log[f'test/auc_pr_{class_names[i].lower()}'] = pr_auc

        ax1.plot([0, 1], [0, 1], 'k--', lw=2)
        ax1.set_xlim([0.0, 1.0])
        ax1.set_ylim([0.0, 1.05])
        ax1.set_xlabel('False Positive Rate', fontsize=10)
        ax1.set_ylabel('True Positive Rate', fontsize=10)
        ax1.set_title('ROC Görbe (Bináris)', fontsize=12, fontweight='bold')
        ax1.legend(loc="lower right", fontsize=9)
        ax1.grid(True, linestyle='--', alpha=0.5)

        ax2.set_xlim([0.0, 1.0])
        ax2.set_ylim([0.0, 1.05])
        ax2.set_xlabel('Recall', fontsize=10)
        ax2.set_ylabel('Precision', fontsize=10)
        ax2.set_title('Precision-Recall Görbe (Bináris)', fontsize=12, fontweight='bold')
        ax2.legend(loc="lower left", fontsize=9)
        ax2.grid(True, linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        metrics_to_log['test/plots/roc_pr_curves'] = wandb.Image(fig_curves)
        plt.close(fig_curves)
        
        cm = confusion_matrix(all_labels, all_preds)
        fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names, ax=ax_cm)
        ax_cm.set_xlabel('AI Predikció', fontsize=11, fontweight='bold')
        ax_cm.set_ylabel('Valóság', fontsize=11, fontweight='bold')
        ax_cm.set_title('Caries Detektor - Confusion Matrix', fontsize=13, pad=20)
        plt.tight_layout()
        
        metrics_to_log['test/plots/confusion_matrix'] = wandb.Image(fig_cm)
        plt.close(fig_cm)
        
        if self.logger and hasattr(self.logger, 'experiment'):
            self.logger.experiment.log(metrics_to_log)
            print("Bináris metrikák, ROC/PR görbe és Confusion Matrix feltöltve a WandB-re!")

        self.test_outputs.clear()

    def configure_optimizers(self):
      optimizer = torch.optim.AdamW(
          filter(lambda p: p.requires_grad, self.parameters()),
          lr=self.cfg.lr,
          weight_decay=self.cfg.weight_decay
      )
      return optimizer

In [8]:
class EvalConfig:
    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    wandb_project: str = 'tooth-caries-classification'
    wandb_run_id = 'v6sk3pso'
    force_download: bool = False

cfg = EvalConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [9]:
env_path = Path('/work/.env')
if not env_path.exists():
    raise FileNotFoundError(f'.env file not found at {env_path}')

load_dotenv(env_path, override=True)

wandb_secret = (os.getenv('WANDB_API_KEY') or '').strip().strip('"').strip("'")
if not wandb_secret:
    raise RuntimeError(f'WANDB_API_KEY not found in {env_path}')

if len(wandb_secret) < 20:
    raise RuntimeError(
        f'Invalid WANDB_API_KEY length ({len(wandb_secret)}). '
        'Expected a classic API key or a wandb_v1 access token from https://wandb.ai/authorize.'
    )

# Support both classic API keys and wandb_v1 access tokens.
try:
    os.environ['WANDB_API_KEY'] = wandb_secret
    wandb.login(key=wandb_secret, relogin=True)
except AuthenticationError:
    if wandb_secret.startswith('wandb_v1_'):
        # Compatibility fallback for SDKs that validate only classic key shapes.
        compat_key = wandb_secret.replace('wandb_v1_', '', 1)
        os.environ['WANDB_API_KEY'] = compat_key
        wandb.login(key=compat_key, relogin=True)
    else:
        raise

print('W&B login successful using WANDB_API_KEY from .env')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nagycsd01 (nagycsd01-university-of-budapest-technology-and-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful using WANDB_API_KEY from .env


In [10]:
datamodule = ToothClassificationDataModule(cfg)
datamodule.setup()

Betöltés memóriába innen: /work/ready_crops/pre_cropped_records.json
🔄 Osztályok összevonása Bináris (Caries vs No Caries) feladatra...
Number of classes: 2
Class weights: tensor([0.5286, 9.2363])
Train samples: 76625
Val samples: 16593
Test samples: 16380


In [ ]:
checkpoint_path = PROJECT_ROOT / 'output' / 'checkpoints' / 'classification' / 'efficientnet-finetune-2 final label-v1.ckpt'

wandb_logger = WandbLogger(
    project=cfg.wandb_project,
    id=cfg.wandb_run_id,
    resume='must'
)

model = LitToothClassifier.load_from_checkpoint(
    str(checkpoint_path),
    cfg=cfg,
    class_weights=datamodule.class_weights
)
model.eval()
model.freeze()

tester = L.Trainer(
    accelerator='auto',
    devices=1,
    precision='16-mixed',
    logger=wandb_logger
)

test_results = tester.test(model, datamodule=datamodule)

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless 

Betöltés memóriába innen: /work/ready_crops/pre_cropped_records.json
🔄 Osztályok összevonása Bináris (Caries vs No Caries) feladatra...
Number of classes: 2
Class weights: tensor([0.5286, 9.2363])


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train samples: 76625
Val samples: 16593
Test samples: 16380


Output()

Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 32. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 28. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

✅ Bináris metrikák, ROC/PR görbe és Confusion Matrix feltöltve a WandB-re!

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/acc          │    0.9763125777244568     │
│          test/f1          │    0.7966457009315491     │
│         test/loss         │    0.09148837625980377    │
└───────────────────────────┴───────────────────────────┘

In [13]:
wandb.finish()

epoch,▁
test/acc,▁
test/auc_pr_caries,▁
test/auc_pr_no_caries,▁
test/auc_roc_caries,▁
test/auc_roc_no_caries,▁
test/f1,▁
test/loss,▁
trainer/global_step,▁▁
epoch,0
lr-AdamW,5e-05
